In [0]:
%pip install --upgrade python-geoclient-moo

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import geoclient

In [0]:
df = spark.read.table('scorecard_fulcrum.geo.midpoint_with_address')

In [0]:
df.describe()

In [0]:
df.head(5)

In [0]:
key = dbutils.secrets.get(scope='ops-secret-scope', key='GEOCLIENT_SUBSCRIPTION_KEY')
if key is not None: 
    print(len(key)) 
else: 
    print('GEOCLIENT_SUBSCRIPTION_KEY is not set')

#make a list of dicts with street number, street name, and borough from the midpoint_with_address table
addresses = spark.sql("select house_number, street, borough from scorecard_fulcrum.geo.midpoint_with_address").toPandas().to_dict('records')

with geoclient.GeoClient(key) as client:
    df = geoclient.batch_geocode_addresses(addresses, client=client)


In [0]:
df = spark.createDataFrame(df)
df.write.mode('overwrite').saveAsTable('scorecard_fulcrum.geo.midpoint_cross_street')
df = spark.read.table('scorecard_fulcrum.geo.midpoint_cross_street')
df.head(5)
df.describe()
df.count()

In [0]:
view = df.select(["address"])